In [1]:
import torch

from utils.event_decoder import EventDecoding
from models.dsa import DSA
from utils.transform import ValTransform
from checkpoints import DT, MT, LT
import librosa
import soundfile as sf

SAMPLE_RATE = 32000
FILEPATH = "XC973025.mp3"

/home/bellafkir/Documents/sa4birds/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
!wget -O XC973025.mp3 https://xeno-canto.org/973025/download

--2026-02-11 10:27:35--  https://xeno-canto.org/973025/download
Resolving xeno-canto.org (xeno-canto.org)... 145.136.250.151
Connecting to xeno-canto.org (xeno-canto.org)|145.136.250.151|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [audio/mpeg]
Saving to: ‘XC973025.mp3’

XC973025.mp3            [ <=>                ]   1.25M  --.-KB/s    in 0.1s    

2026-02-11 10:27:35 (10.7 MB/s) - ‘XC973025.mp3’ saved [1311882]



In [2]:
def build_model(cfg):
    if cfg.network.classifier == "DSA":
        base_model = DSA(cfg)
    else:
        raise NotImplementedError

    base_model.to(cfg.train.device)
    return base_model

def load_model(checkpoint):
    state_dict = torch.load(f"{checkpoint}/models/model.pth")
    state_dict, config = state_dict['state_dict'], state_dict['config']
    model = build_model(config)
    model.load_state_dict(state_dict)
    model.eval()
    return model, config

In [23]:

# down_task = "HSN"
# change it if you want to use a different model
ckpt = LT.get('ALL', [])[0]

model, cfg = load_model(ckpt)
transform = ValTransform(
        config=cfg,
        train=False,
        event_decoder=EventDecoding(extracted_interval=cfg.event_decoder.val.extracted_interval),
)
label_map = { v:k for k,v in cfg.train.label_map.items()}

file_info = sf.info(FILEPATH)
sr = file_info.samplerate
audio, sr = sf.read(FILEPATH, start=0, stop=7*sr)


if audio.ndim != 1:
    audio = audio.swapaxes(1, 0)
    audio = librosa.to_mono(audio)
if sr != SAMPLE_RATE:
    audio = librosa.resample(audio, orig_sr=sr, target_sr=SAMPLE_RATE)
    sr = SAMPLE_RATE

fbank_features = transform.get_spectrogram(torch.tensor(audio).unsqueeze(0).unsqueeze(0).float())
fbank_features = transform.pad(fbank_features)
fbank_features = (fbank_features - cfg.frontend.mean) / ( cfg.frontend.std * 2)

with torch.no_grad():
    preds = model(fbank_features.to(cfg.train.device))
    top1 = preds.topk(1)
    print(f"{label_map[int(top1.indices[0])]}: {top1.values[0][0]}")

redkit1: 0.957703173160553
